#### *En este JuypterNotebook se procesara el dataset "Aeropuertos de Argentina". El mismo sera modificado en un nuevo archivo*

##### Importo la libreria csv y las direcciones de los archivos a utilizar

In [1]:
import csv
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve().parent.parent))
from modules.paths import AR_AIRPORTS_DATA,AR_AIRPORTS_DATA_MODIFIED,AR_DATA
import modulo as airport_functions
import re


#### Esta funcion abre el archivo de AR_AIRPORTS_DATA que contiene info de los Aeropuertos en Argentina, e itera por cada fila del mismo para obtener los datos necesarios y generar los campos a agregar en el archivo modificado de Aeropuertos (AR_AIRPORTS_MODIFIED_DATA)

In [2]:
def process_datasets():
    try:
        with (
            AR_AIRPORTS_DATA.open(mode='r',encoding='utf-8')as read_file,
            AR_AIRPORTS_DATA_MODIFIED.open(mode='w',encoding='utf-8',newline='')as write_file
        ):
            reader=csv.reader(read_file)
            writer=csv.writer(write_file)

            new_header=next(reader)
            new_header.append('elevation_name')#agrego al encabezado la fila elevation_name
            
            new_header.append('prov_name')#agrego al encabezado la fila prov_name
            writer.writerow(new_header)#escribo en el archivo modificado lo que habia en el original, mas las nuevas filas
        
            dic_prov_name=[]
            dic_prov_name=airport_functions.build_dic_prov(AR_DATA) #construyo diccionario con nombre de ciudad y prov a la que pertenece
            
            # Defino constantes que identifican las columnas de elevacion y municipalidad
            elevation_row=6
            municipality_row=13
        
            for row in reader: 
                
                elevation=(row[elevation_row])##guardo la elevacion, todavia sin covertir a int porque puede ser campo vacio

                category=airport_functions.value_category(elevation) #invoco a la funcion que me devuelve la cat del aeropuerto

                municipality_airport = airport_functions.remove_accents(row[municipality_row])

                # Usar expresión regular para eliminar '/', dividir por '('
                municipality_airport_modified = re.split(r'[()/]', municipality_airport)

                # Eliminar espacios en blanco de cada parte resultante y eliminar elementos vacíos
                municipality_airport_modified = [part.strip() for part in municipality_airport_modified if part.strip()]

            
                prov='N/A' #si no se encuentra la provincia
                for dic in dic_prov_name: #recorro la lista de diccionarios buscando la provincia de la municipalidad actual
                    
                    municipality_in_ardata=airport_functions.remove_accents(dic['City'])
                    if municipality_in_ardata in municipality_airport_modified:
                        prov=dic['Prov']
                        break



                row.append(category)

                row.append(prov)

            
                
                writer.writerow(row)
    except FileNotFoundError:
        print('Error, el archivo AR_AIRPORTS_DATA no fue encontrado')




##### Invoco la función:

In [3]:
process_datasets()